# NB12e - UNOSAT Window Sensitivity Analysis

> Copyright (C) 2024-2026 Marco Heinzen - SPDX-License-Identifier: AGPL-3.0-or-later
> Part of the Master Thesis "Building Damage Assessment with Multimodal Satellite Time Series and Machine Learning in the Russia-Ukraine War 2022-2026"
> Code hosted at https://github.com/marcoheinzen/bda
> Parts of this code were written or improved with the assistance of Claude (Anthropic); all other code, and the concept, research, architecture, design, execution, testing and validation throughout, are the author's work.

Sensitivity analysis (classical sense): vary the observation-window cutoff relative to the
per-city UNOSAT assessment date (input parameter), hold features/model/CV fixed, and measure
the response of mean-of-folds AUC (output). Answers supervisor review 2026-07 (Scholz):
"Sensitivitaetsanalyse bzgl. UNOSAT-Daten und dem kurzen Zeitfenster vs laengere Zeitdauer des Datensatzes".
Arms: cutoff = unosat_last + delta, delta in {0,30,90,180,365,730} days, plus FULL window,
plus a scene-count-matched FULL control (separates window LENGTH from scene COUNT).
Features recomputed from the per-building scene tables with one shared code path for all arms.
Model: LightGBM-300 (class_weight balanced, native NaN); CV: frozen 5-fold GroupKFold-by-city.

In [ ]:
import io
import json
import os
import time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
import lightgbm as lgb

STACK = "/mnt/f/PROJECTS/masterthesis/data_stack"
DSET = STACK + "/dataset/v1"
V2 = STACK + "/dataset/V2"
A2 = "/mnt/f/PROJECTS/masterthesis/gdrive/masterthesis/md-files/thesis_draft/A2_unosat_snapshot_dates.csv"
GKF = STACK + "/groupkfold_assignments.json"
OUT = "/mnt/f/PROJECTS/masterthesis/gdrive/masterthesis/results/nb12"
os.makedirs(OUT, exist_ok=True)
TS = time.strftime("%Y%m%d_%H%M%S")
SEED = 42
DELTAS = [0, 30, 90, 180, 365, 730]
print("NB12e start", TS)

## Cell 1 - City dates (A2) and frozen fold assignment

In [ ]:
a2 = pd.read_csv(A2, encoding="utf-8-sig")
a2["battle_start"] = pd.to_datetime(a2["battle_start"])
a2["unosat_last"] = pd.to_datetime(a2["unosat_last"])
dates = a2.set_index("city")[["battle_start", "unosat_last"]]
print(dates)

gk = json.load(open(GKF))
print("assignment file:", gk.get("fold_mode"), "- headline protocol is 5-fold GroupKFold-by-city (sklearn, deterministic); applied below")

## Cell 2 - Load per-building scene tables (coherence pairs + CARD backscatter)

In [ ]:
def load_scenes(prefix, cols, datecol):
    parts = []
    for t in ["t0", "t1", "t2"]:
        f = DSET + "/" + prefix + "_" + t + ".parquet"
        parts.append(pq.read_table(f, columns=cols).to_pandas())
    df = pd.concat(parts, ignore_index=True)
    df["obs_date"] = pd.to_datetime(df[datecol], format="%Y%m%d")
    return df

coh = load_scenes("bda_scene_coh", ["building_id", "city", "date2", "s1__coh_vv_mean", "s1__coh_vh_mean"], "date2")
card = load_scenes("bda_scene_card", ["building_id", "city", "date", "s1__vv_mean", "s1__vh_mean"], "date")
print("coh rows:", len(coh), "| card rows:", len(card))
print("coh cities:", coh["city"].nunique(), "| card cities:", card["city"].nunique())

## Cell 3 - Baseline statistics (pre-battle) and shared accumulator code path

In [ ]:
def add_city_dates(df):
    df = df.join(dates, on="city")
    return df[df["battle_start"].notna()]

coh = add_city_dates(coh)
card = add_city_dates(card)

def baseline_stats(df, vals):
    pre = df[df["obs_date"] < df["battle_start"]]
    agg = pre.groupby("building_id")[vals].agg(["mean", "std", "count"])
    agg.columns = ["_".join(c) for c in agg.columns]
    return agg

coh_base = baseline_stats(coh, ["s1__coh_vv_mean", "s1__coh_vh_mean"])
card_base = baseline_stats(card, ["s1__vv_mean", "s1__vh_mean"])
print("buildings with coh baseline:", len(coh_base), "| card baseline:", len(card_base))

def accumulators(df, base, vals, tag, cutoff_by_city=None, match_counts=None):
    post = df[df["obs_date"] >= df["battle_start"]].copy()
    if cutoff_by_city is not None:
        post = post.join(cutoff_by_city.rename("cutoff"), on="city")
        post = post[post["obs_date"] <= post["cutoff"]]
    if match_counts is not None:
        rng = np.random.default_rng(SEED)
        post = post.sample(frac=1.0, random_state=SEED)
        post["_k"] = post.groupby("building_id").cumcount()
        post = post.join(match_counts.rename("_n"), on="building_id")
        post = post[post["_k"] < post["_n"].fillna(0)]
    post = post.join(base, on="building_id")
    feats = {}
    for v in vals:
        mu, sd = post[v + "_mean"], post[v + "_std"]
        z = (post[v] - mu) / sd.replace(0.0, np.nan)
        post["_z_" + v] = z
    g = post.groupby("building_id")
    out = pd.DataFrame(index=g.size().index)
    out[tag + "_n_scenes"] = g.size()
    for v in vals:
        zc = "_z_" + v
        out[tag + "_zmin_" + v[-7:]] = g[zc].min()
        out[tag + "_dropcnt_" + v[-7:]] = g[zc].apply(lambda s: (s < -1.0).sum())
    return out

## Cell 4 - Labels (V2 buildings) and feature assembly per window arm

In [ ]:
lb = []
for t in ["t0", "t1", "t2"]:
    tf = pq.read_table(V2 + "/bda_buildings_" + t + ".parquet")
    cols = [c for c in ["building_id", "city", "damage_binary"] if c in tf.schema.names]
    lb.append(tf.select(cols).to_pandas())
labels = pd.concat(lb, ignore_index=True).drop_duplicates("building_id")
labels = labels[labels["city"].isin(dates.index)]
# raw inventory carries -1 = unknown (pre-convention coding); thesis convention: only ==1 is positive
labels["damage_binary"] = (labels["damage_binary"] == 1).astype(int)
print("labelled buildings (inventory):", len(labels), "| positives:", int(labels["damage_binary"].sum()))

FEATCOLS = None

def build_arm(name, cutoff_by_city, match_counts=None):
    global FEATCOLS
    fc = accumulators(coh, coh_base, ["s1__coh_vv_mean", "s1__coh_vh_mean"], "coh", cutoff_by_city, match_counts)
    fd = accumulators(card, card_base, ["s1__vv_mean", "s1__vh_mean"], "card", cutoff_by_city, match_counts)
    X = labels.join(fc, on="building_id").join(fd, on="building_id")
    featcols = [c for c in X.columns if ("zmin" in c or "dropcnt" in c)]
    if FEATCOLS is None:
        FEATCOLS = featcols
    scn = X[[c for c in X.columns if c.endswith("_n_scenes")]].mean().to_dict()
    return X, featcols, scn

## Cell 5 - Frozen GroupKFold evaluation, one arm per window setting

In [ ]:
def evaluate(X, featcols):
    X = X.copy()
    y = X["damage_binary"].astype(int).values
    F = X[featcols].astype(float).values
    groups = X["city"].values
    oof = np.full(len(X), np.nan)
    fold_aucs = []
    for tr, te in GroupKFold(n_splits=5).split(F, y, groups):
        clf = lgb.LGBMClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, n_jobs=-1, verbosity=-1)
        clf.fit(F[tr], y[tr])
        p = clf.predict_proba(F[te])[:, 1]
        oof[te] = p
        if len(set(y[te])) == 2:
            fold_aucs.append(roc_auc_score(y[te], p))
    percity = {}
    for c, sub in X.assign(p=oof).groupby("city"):
        if sub["damage_binary"].nunique() == 2:
            percity[c] = roc_auc_score(sub["damage_binary"], sub["p"])
    return float(np.mean(fold_aucs)), float(np.std(fold_aucs)), percity, oof, X.index

rows = []
percity_rows = []
counts_at_0 = None

# fixed population: buildings with at least one usable feature under the FULL window,
# so all arms share one denominator and differ only in features
X_full, FEATCOLS, _ = build_arm("FULL", None)
POP = set(X_full.loc[X_full[FEATCOLS].notna().any(axis=1), "building_id"])
print("analysis population (any feature under FULL):", len(POP),
      "| positives:", int(X_full[X_full["building_id"].isin(POP)]["damage_binary"].sum()))

def run_arm(name, X):
    X = X[X["building_id"].isin(POP)]
    m, s, pc, oof, idx = evaluate(X, FEATCOLS)
    nsc = float(np.nanmean(X[[c for c in X.columns if c.endswith("_n_scenes")]].mean(axis=1)))
    rows.append({"arm": name, "mean_folds_auc": m, "sd_folds": s,
                 "mean_scenes_per_building": nsc, "n_buildings": len(X)})
    for c, v in pc.items():
        percity_rows.append({"arm": name, "city": c, "auc": v})
    print(name, "mean-of-folds AUC = %.4f +- %.4f | scenes/bldg %.1f" % (m, s, nsc))
    return X

for d in DELTAS:
    cut = dates["unosat_last"] + pd.Timedelta(days=int(d))
    X, _, _ = build_arm("d+" + str(d), cut)
    X = run_arm("d+" + str(d), X)
    if d == 0:
        counts_at_0 = X.set_index("building_id")[["coh_n_scenes", "card_n_scenes"]]
run_arm("FULL", X_full)

# scene-count-matched FULL control (separates length from count)
mc = counts_at_0[["coh_n_scenes"]].rename(columns={"coh_n_scenes": "n"})["n"]
mcc = counts_at_0[["card_n_scenes"]].rename(columns={"card_n_scenes": "n"})["n"]
fc = accumulators(coh, coh_base, ["s1__coh_vv_mean", "s1__coh_vh_mean"], "coh", None, mc)
fd = accumulators(card, card_base, ["s1__vv_mean", "s1__vh_mean"], "card", None, mcc)
Xm = labels.join(fc, on="building_id").join(fd, on="building_id")
run_arm("FULL_count_matched_to_d0", Xm)

## Cell 6 - City-bootstrap CI for FULL minus capped(d0); outputs

In [ ]:
curve = pd.DataFrame(rows)
pcdf = pd.DataFrame(percity_rows)
piv = pcdf.pivot(index="city", columns="arm", values="auc")
rng = np.random.default_rng(SEED)
if "d+0" in piv.columns and "FULL" in piv.columns:
    both = piv[["FULL", "d+0"]].dropna()
    diffs = both["FULL"] - both["d+0"]
    bs = [diffs.sample(len(diffs), replace=True, random_state=int(rng.integers(1e9))).mean() for _ in range(2000)]
    ci = (float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5)))
    print("FULL - d0 per-city mean diff = %.4f, 95%% CI [%.4f, %.4f], n=%d cities" % (diffs.mean(), ci[0], ci[1], len(diffs)))
    curve.attrs["full_minus_d0_ci"] = ci

curve.to_csv(OUT + "/nb12e_window_sensitivity_curve_" + TS + ".csv", index=False)
pcdf.to_csv(OUT + "/nb12e_per_city_auc_by_window_" + TS + ".csv", index=False)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 5))
sweep = curve[curve["arm"].str.startswith("d+")].copy()
sweep["x"] = sweep["arm"].str.replace("d+", "", regex=False).astype(float)
ax.errorbar(sweep["x"], sweep["mean_folds_auc"], yerr=sweep["sd_folds"], marker="o", capsize=3, label="capped at unosat_last + delta")
full = curve[curve["arm"] == "FULL"].iloc[0]
ax.axhline(full["mean_folds_auc"], ls="--", color="tab:red", label="full window (%.3f)" % full["mean_folds_auc"])
mfull = curve[curve["arm"] == "FULL_count_matched_to_d0"].iloc[0]
ax.axhline(mfull["mean_folds_auc"], ls=":", color="tab:green", label="full, scene-count matched to d0 (%.3f)" % mfull["mean_folds_auc"])
ax.set_xscale("symlog", linthresh=30)
ax.set_xlabel("window extension beyond UNOSAT assessment date [days]")
ax.set_ylabel("mean-of-folds AUC (5-fold GroupKFold-by-city)")
ax.set_title("NB12e: sensitivity of cross-city AUC to the observation-window cutoff\n(SAR accumulator features, LightGBM-300, frozen folds)")
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
fig.savefig(OUT + "/nb12e_sensitivity_diagram_" + TS + ".png", dpi=200)
print("outputs written to", OUT, "with timestamp", TS)
print("NB12e done")